# FASTT — inference cost (Table 4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/fastt/blob/main/notebooks/timing.ipynb)

Where the time goes when an ensemble of speech recognizers serves one clip: in each expert's encoder or in its
decoder, and what choosing the expert from the encoder outputs, before anything is decoded, saves against decoding
every expert and then choosing. Measured on VoxPopuli.

**What is measured.** Each expert's encoder and decoder, separately, per clip at batch size 1, on the same 32 random
clips (after 3 untimed warm-up clips), at greedy decoding and at beam widths 5 and 20. Everything else is composed
from those clocks and the main run's record: decoding every expert pays every encoder and every decoder; FASTT pays
every encoder, the selector, and the decoder of the expert it picked, weighted by how often it picks each. The
selectors' own training and inference times come from the main run's 5×2 folds.

| Level | What runs | Needs |
|---|---|---|
| **1** | every table, recomposed from the clocks the paper measured (stored on the Hub) | a CPU, seconds |
| **2** | the clocks measured again on this machine, then every table | a CUDA GPU, the experts' toolkits (`extract`'s groups A and C) |

Absolute times depend on the GPU; the ratios, and so the conclusion, much less.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("fastt") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "fastt[paper] @ git+https://github.com/huseyin-karaca/fastt.git@main"], check=True)

In [2]:
from fastt.timing import Timing

LEVEL = 1          # 1 or 2 (see above)

t = Timing("voxpopuli", level=LEVEL)

VoxPopuli: level 1 — clocks read from the Hub
experts  ESPnet Conformer-RNN-T, ESPnet Transformer, data2vec-audio-base
FASTT-Boosted picks ESPnet Conformer-RNN-T 35%, ESPnet Transformer 0%, data2vec-audio-base 65%; selector 1.5 µs per clip


## 1. The clocks

In [3]:
t.clocks()

model,decoding,encoder_seconds,decoder_seconds,total_seconds,audio_seconds,rtf,dec_over_enc,wer,n_clips
espnet_transducer,greedy,0.0101,0.0358,0.0459,10.2923,0.0045,3.5445,0.1993,32
espnet_transducer,beam5,0.0084,0.3061,0.3145,10.2923,0.0306,36.4326,0.1931,32
espnet_transducer,beam20,0.0088,3.1960,3.2048,10.2923,0.3114,363.3695,0.1907,32
espnet_transformer,greedy,0.0070,0.0002,0.0071,10.2923,0.0007,0.0255,0.2509,32
espnet_transformer,beam5,0.0063,0.6140,0.6203,10.2923,0.0603,96.6941,0.2140,32
espnet_transformer,beam20,0.0064,0.6869,0.6933,10.2923,0.0674,107.2908,0.2165,32
data2vec,greedy,0.0195,0.0003,0.0198,10.2923,0.0019,0.0140,0.2042,32
data2vec,beam5,0.0065,0.0074,0.0138,10.2923,0.0013,1.1406,0.2054,32
data2vec,beam20,0.0065,0.0234,0.0298,10.2923,0.0029,3.6190,0.2042,32


In [4]:
t.compose()
t.decoding()

expert,encode_ms,greedy_decode_ms,greedy_x_encoder,greedy_wer,beam5_decode_ms,beam5_x_encoder,beam5_wer,beam20_decode_ms,beam20_x_encoder,beam20_wer
ESPnet Conformer-RNN-T,9.1,35.8,3.94,0.199,306.1,33.63,0.193,3196.0,351.09,0.191
ESPnet Transformer,6.6,0.2,0.03,0.251,614.0,93.39,0.214,686.9,104.48,0.216
data2vec-audio-base,10.8,0.3,0.03,0.204,7.4,0.68,0.205,23.4,2.16,0.204


## 2. Decoding every expert, or choosing first (Table 4b)

Per beam width: each expert alone, decoding every expert and then choosing, choosing from the encoders and decoding
one expert (FASTT), and the floor of always decoding the cheapest expert.

In [5]:
t.ensemble()

Ensemble cost per clip — VoxPopuli
x_after_decoding: total relative to decoding every expert at the same beam width
  [greedy]
                      system  encode_ms  decode_ms  selector_ms  total_ms  x_best_single  x_after_decoding
      ESPnet Conformer-RNN-T       10.1       35.8        0.000      45.9           6.43              0.63
          ESPnet Transformer        7.0        0.2        0.000       7.1           1.00              0.10
         data2vec-audio-base       19.5        0.3        0.000      19.8           2.77              0.27
     decode all, then choose       36.6       36.3        0.001      72.9          10.20              1.00
choose from encoders (FASTT)       36.6       12.8        0.001      49.4           6.91              0.68
       cheapest decoder only       36.6        0.2        0.001      36.8           5.15              0.50
  [beam5]
                      system  encode_ms  decode_ms  selector_ms  total_ms  x_best_single  x_after_decoding
      ESPnet Conformer-RNN-T        8.4      306.1        0.000     314.5          22.74              0.33
          ESPnet Transformer        6.3      614.0        0.000     620.3          44.85              0.65
         data2vec-audio-base        6.5        7.4        0.000      13.8           1.00              0.01
     decode all, then choose       21.2      927.5        0.001     948.7          68.60              1.00
choose from encoders (FASTT)       21.2      112.2        0.001     133.4           9.65              0.14
       cheapest decoder only       21.2        7.4        0.001      28.6           2.07              0.03
  [beam20]
                      system  encode_ms  decode_ms  selector_ms  total_ms  x_best_single  x_after_decoding
      ESPnet Conformer-RNN-T        8.8     3196.0        0.000    3204.8         107.37              0.82
          ESPnet Transformer        6.4      686.9        0.000     693.3          23.23              0.18
         data2vec-audio-base        6.5       23.4        0.000      29.8           1.00              0.01
     decode all, then choose       21.7     3906.2        0.001    3927.9         131.59              1.00
choose from encoders (FASTT)       21.7     1136.4        0.001    1158.0          38.80              0.29
       cheapest decoder only       21.7       23.4        0.001      45.0           1.51              0.01

In [6]:
t.savings()

greedy   decoding is 50% of decode-then-choose; choosing from the encoders saves 23.5 ms per clip (32%)
beam5    decoding is 98% of decode-then-choose; choosing from the encoders saves 815.3 ms per clip (86%)
beam20   decoding is 99% of decode-then-choose; choosing from the encoders saves 2769.9 ms per clip (70%)


decoding,decode_share_after,total_after_ms,total_before_ms,saved_ms,saved_fraction
greedy,0.498,72.9,49.4,23.5,0.323
beam5,0.978,948.7,133.4,815.3,0.859
beam20,0.994,3927.9,1158.0,2769.9,0.705


## 3. Every method at greedy decoding (Table 4c)

Encode, decode, selector inference and end-to-end time per clip for every method of Table 3, and the selector's
training time on one 5×2 fold.

In [7]:
t.table4()

system,encode_ms,decode_ms,selector_us,end_to_end_ms,train_selector_s,x_fastest_member
ESPnet Conformer-RNN-T,10.1,35.8,0.0,45.9,0.00,6.43
ESPnet Transformer,7.0,0.2,0.0,7.1,0.00,1.00
data2vec-audio-base,19.5,0.3,0.0,19.8,0.00,2.77
Best-Single-Model,36.6,35.8,0.0,72.5,0.00,10.13
Random-Ensemble,36.6,12.1,0.0,48.8,0.00,6.82
ROVER (weighted),36.6,36.3,225.6,73.1,0.00,10.23
ROVER (confidence-weighted),36.6,36.3,42.9,73.0,0.00,10.20
CN-MBR,36.6,20.1,46.4,56.7,2.63,7.94
Score-Fusion,36.6,11.6,22.9,48.2,0.00,6.75
Hard-Selector (plain CE),36.6,29.6,0.5,66.2,0.09,9.26


## 4. Against the paper's record, and the record

At level 1 every table is identical to the one the paper reports; at level 2 the differences are this machine's.

In [8]:
t.compare()

table,rows,identical,max_abs_diff
per_model,9,True,0
decoding,3,True,0
ensemble_cost,18,True,0
savings,3,True,0
table4,21,True,0


In [9]:
t.save();

wrote timing_voxpopuli.json
